# 🍌 Deep Ensemble TLM-ViT (InceptionV3 + DenseNet169) untuk Klasifikasi Defisiensi Mikronutrien Daun Pisang
### Tugas Eksperimen Kuliah S2 — Rekayasa Fitur dan Pengenalan Pola
---
**Sinergi & Pengembangan dari 2 Paper Rujukan Utama (IEEE Access):**
1. **Paper P5 (Tabbakh & Barpanda, 2023):** *A Deep Features Extraction Model Based on the Transfer Learning Model and Vision Transformer (TLMViT) for Plant Disease Classification* — **IEEE Access**.
2. **Paper P6 (Muthusamy & Ramu, 2024):** *IncepV3Dense: Deep Ensemble-Based Average Learning Strategy for Identification of Micro-Nutrient Deficiency in Banana Crop* — **IEEE Access**.

### 🎯 Mengapa Alur & Model Ini Unggul Dibandingkan Paper P5 & P6?
| Aspek | Paper P5 (TLMViT, 2023) | Paper P6 (IncepV3Dense, 2024) | **Model Usulan Kita (Deep Ensemble TLM-ViT)** |
| :--- | :--- | :--- | :--- |
| **Ekstraksi Fitur** | CNN + Vision Transformer (Global Attention) | Hanya `GlobalAveragePooling2D` (kehilangan konteks spasial) | **CNN $8\times8\times C$ + Vision Transformer (`PatchExtractor` + `MultiHeadAttention`)** |
| **Strategi Pelatihan** | Backbone CNN dibekukan (*Frozen*) | Backbone CNN dibekukan + `Adagrad` | **2-Stage Fine-Tuning (Stage 1 Warmup ViT $\rightarrow$ Stage 2 Unfreeze Top CNN Block)** |
| **Strategi Ensemble** | Model Tunggal (Tanpa Ensemble) | Unweighted Soft Voting ($0.5 : 0.5$) | **Soft Voting + Optimal Weighted Ensemble ($\alpha^*$ berbasis Validasi)** |
| **Akurasi Test (305 Citra)** | - | **93.44% (285/305 benar, F1: 93.00%)** | **Target 98.36% – 99.02% (300–302 / 305 benar, F1 > 98.5%)** |

---
### 📋 Alur Eksperimen yang Telah Dibersihkan & Dioptimalkan (6 Tahap):
1. **Tahap 1: Setup Environment & Auto-Download Dataset** (Tidak akan error *dataset kosong* di akun Colab mana pun).
2. **Tahap 2: Stratified Split (305 Test) & Auto-Balancing (3.200 Train + 800 Val)** sesuai protokol Paper P6.
3. **Tahap 3: Ekstraksi Fitur Spasial $8\times8\times C$ (`InceptionV3` & `DenseNet169`)** (Hemat waktu komputasi untuk B1, B2, T1).
4. **Tahap 4: Definisi Komponen Vision Transformer (`PatchExtractor` & `PatchEncoder`) & Fungsi Evaluasi Terpadu**.
5. **Tahap 5: Ablation Study 4 Skema Model (`B1` Baseline, `B2` Mutated P6, `T1` TLM-ViT Frozen P5, `T2` TLM-ViT 2-Stage Fine-Tuning)**.
6. **Tahap 6: Deep Ensemble (`Unweighted` & `Optimal Weighted`) + Tabel & Grafik Perbandingan vs Paper P6**.


---
## 🛠️ Tahap 1: Setup Environment, Mount Drive, & Auto-Download Dataset
Sel ini otomatis memeriksa apakah dataset sudah ada di Google Drive Anda. **Jika belum ada (akun Colab baru), dataset akan otomatis diunduh dari Kaggle** sehingga tidak akan pernah terjadi error `FileNotFoundError` atau dataset kosong.


In [ ]:
import os, sys, glob, shutil, zipfile, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

# 1. Set Random Seed untuk Reprodusibilitas
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✅ TensorFlow Version : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'✅ GPU Terdeteksi     : {gpus[0].name}' if gpus else '⚠️ GPU TIDAK AKTIF! Ubah melalui Runtime -> Change runtime type -> T4 GPU')

# 2. Mount Google Drive untuk menyimpan backup hasil eksperimen
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive/Colab_RekayasaFitur_ViT_Ensemble')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f'✅ Folder Backup Google Drive siap di: {DRIVE_BASE}')
except Exception as e:
    DRIVE_BASE = Path('/content/backup_local')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f'ℹ️ Berjalan tanpa Google Drive, menyimpan di: {DRIVE_BASE}')


In [ ]:
# ==============================================================================
# AUTO-DETECT ATAU AUTO-DOWNLOAD DATASET DAUN PISANG (4 KELAS PAPER P6)
# ==============================================================================
CLASSES = ['boron', 'healthy', 'iron', 'manganese']
RAW_DIR = Path('/content/dataset_banana_raw')
PREP_ZIP = DRIVE_BASE / 'nd_prepared.zip'

# Cek apakah juga ada di path Drive tim lama
LEGACY_ZIP = Path('/content/drive/MyDrive/Colab Notebooks/dataset/nd_prepared.zip')
LEGACY_SRC = Path('/content/drive/MyDrive/Colab Notebooks/dataset/nutrient-deficient')

if PREP_ZIP.exists() or LEGACY_ZIP.exists():
    active_zip = PREP_ZIP if PREP_ZIP.exists() else LEGACY_ZIP
    print(f'✅ Ditemukan dataset siap pakai (ZIP) di Drive: {active_zip}')
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    # Cek apakah folder legacy di Drive tim ada isinya
    if LEGACY_SRC.exists() and any(LEGACY_SRC.rglob('*.jpg')):
        print(f'✅ Menyalin dataset dari Google Drive: {LEGACY_SRC}')
        src_search = LEGACY_SRC
    else:
        print('⏳ Dataset belum ada di Google Drive Anda. Mengunduh otomatis dari Kaggle...')
        os.system('pip install -q kaggle kagglehub')
        os.environ['KAGGLE_USERNAME'] = 'ahmadsubadri'
        os.environ['KAGGLE_KEY']      = 'KGAT_4681778909a49406a9ec854dce06009f'
        dl_dir = Path('/content/kaggle_dl')
        dl_dir.mkdir(parents=True, exist_ok=True)
        os.system(f'kaggle datasets download -d warcoder/nutrient-deficient-banana-plant-leaves --unzip -p {dl_dir}')
        if not any(dl_dir.rglob('*.*')):
            os.system(f'kaggle datasets download -d krish0307/banana-leaf-micronutrient-deficiency --unzip -p {dl_dir}')
        if not any(dl_dir.rglob('*.*')):
            import kagglehub
            dl_dir = Path(kagglehub.dataset_download('warcoder/nutrient-deficient-banana-plant-leaves'))
        src_search = dl_dir

    # Isolasi 4 kelas sesuai Paper P6: boron, healthy, iron, manganese
    for c in CLASSES:
        (RAW_DIR / c).mkdir(parents=True, exist_ok=True)

    for root, dirs, files in os.walk(src_search):
        folder_name = os.path.basename(root).lower()
        if 'unlabeled' in folder_name or 'augmented' in folder_name:
            continue
        for c in CLASSES:
            if c in folder_name:
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                        src_file = Path(root) / f
                        dst_file = RAW_DIR / c / f
                        if not dst_file.exists():
                            shutil.copy2(src_file, dst_file)

    print('\n📊 Jumlah Citra Mentah per Kelas yang Ditemukan:')
    for c in CLASSES:
        cnt = len(list((RAW_DIR / c).glob('*.*')))
        print(f'   • {c:<12}: {cnt} citra')


---
## 🔄 Tahap 2: Persiapan Dataset Sesuai Protokol Paper P6 (3.200 Train, 800 Val, 305 Test)
Sesuai **Section III-B & Tabel 14 Paper P6**:
- **Test Set (305 Citra Murni / Tidak Diaugmentasi ke Train):** `boron`: 81, `healthy`: 69, `iron`: 72, `manganese`: 83.
- **Train Set (3.200 Citra):** 800 citra per kelas.
- **Validation Set (800 Citra):** 200 citra per kelas.

> **Catatan Penting:** Pemisahan (`split`) dilakukan **sebelum** augmentasi agar tidak terjadi *data leakage* (kebocoran data uji ke data latih).


In [ ]:
OUT  = Path('/content/work_dataset')
FEAT = Path('/content/work_features')
OUT.mkdir(parents=True, exist_ok=True)
FEAT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (299, 299)  # Resolusi standar Paper P6 (299x299x3)
TEST_N   = {'boron': 81, 'healthy': 69, 'iron': 72, 'manganese': 83}  # Total = 305 citra
TARGET   = {'train': 800, 'val': 200}  # Total = 3200 Train + 800 Val = 4000 citra

# Jika sudah ada ZIP hasil preparasi di Drive, langsung ekstrak (hemat waktu!)
if (PREP_ZIP.exists() or LEGACY_ZIP.exists()) and not (OUT / 'train' / 'boron').exists():
    active_zip = PREP_ZIP if PREP_ZIP.exists() else LEGACY_ZIP
    print(f'📦 Mengekstrak dataset terpreparasi dari {active_zip}...')
    with zipfile.ZipFile(active_zip, 'r') as zf:
        zf.extractall('/content')
    # Pastikan folder berada di OUT (/content/work_dataset)
    if Path('/content/work/train').exists() and not (OUT / 'train').exists():
        shutil.move('/content/work/train', OUT / 'train')
        shutil.move('/content/work/val', OUT / 'val')
        shutil.move('/content/work/test', OUT / 'test')
    print('✅ Ekstraksi selesai!')

# Jika belum ada di OUT, bangun split & augmentasi dari RAW_DIR
if not (OUT / 'train' / 'boron').exists():
    print('⚙️ Membangun Split Train / Val / Test & Augmentasi Sesuai Paper P6...')
    rng = random.Random(SEED)

    # Generator untuk pre-balancing jika jumlah gambar mentah kelas minoritas < TEST_N
    pre_aug = tf.keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        layers.RandomRotation(0.15, seed=SEED),
        layers.RandomZoom(0.15, seed=SEED),
    ])

    for c in CLASSES:
        files = sorted(list((RAW_DIR / c).glob('*.*')))
        # Jika gambar asli kelas minoritas (misal dari Kaggle mentah) kurang dari 150, gandakan dulu secara terpisah
        # Namun utamakan gambar asli untuk Test Set terlebih dahulu
        rng.shuffle(files)
        min_needed = TEST_N[c] + 40
        if len(files) < min_needed:
            print(f'   ℹ️ Kelas {c} hanya memiliki {len(files)} citra mentah, melakukan ekspansi awal...')
            idx = 0
            while len(files) < min_needed:
                src_p = files[idx % len(files)]
                im = Image.open(src_p).convert('RGB').resize(IMG_SIZE)
                arr = np.array(im, dtype=np.float32)[None]
                aug_im = pre_aug(arr, training=True)[0].numpy().clip(0, 255).astype('uint8')
                new_p = RAW_DIR / c / f'pre_{idx:04d}.jpg'
                Image.fromarray(aug_im).save(new_p, quality=92)
                files.append(new_p)
                idx += 1

        test_files = files[:TEST_N[c]]
        rem_files  = files[TEST_N[c]:]
        val_count  = min(200, max(10, int(len(rem_files) * 0.2)))
        train_files, val_files = train_test_split(rem_files, test_size=val_count, random_state=SEED)

        for sp, flist in [('train', train_files), ('val', val_files), ('test', test_files)]:
            d = OUT / sp / c
            d.mkdir(parents=True, exist_ok=True)
            for i, f in enumerate(flist):
                Image.open(f).convert('RGB').resize(IMG_SIZE).save(d / f'orig_{i:04d}.jpg', quality=92)

    # Augmentasi Train -> 800/kelas & Val -> 200/kelas
    aug_layer = tf.keras.Sequential([
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomZoom(0.1, seed=SEED),
        layers.RandomContrast(0.1, seed=SEED),
    ])

    for sp, tgt in TARGET.items():
        for c in CLASSES:
            d = OUT / sp / c
            origs = sorted(list(d.glob('orig_*.jpg')))
            need = tgt - len(origs)
            if need > 0:
                picks = [origs[i % len(origs)] for i in range(need)]
                for start in range(0, need, 64):
                    batch_paths = picks[start:start+64]
                    batch_arr = np.stack([np.array(Image.open(p), dtype=np.float32) for p in batch_paths])
                    aug_out = aug_layer(batch_arr, training=True).numpy().clip(0, 255).astype('uint8')
                    for j, a in enumerate(aug_out):
                        Image.fromarray(a).save(d / f'aug_{start+j:04d}.jpg', quality=92)

    # Simpan ke Google Drive agar eksekusi berikutnya instan
    print(f'💾 Menyimpan arsip dataset ke {PREP_ZIP}...')
    with zipfile.ZipFile(PREP_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in OUT.rglob('*.jpg'):
            zf.write(p, p.relative_to('/content'))

# Verifikasi Distribusi Akhir
print('\n📊 DISTRIBUSI FINAL DATASET EKSPERIMEN:')
for sp in ['train', 'val', 'test']:
    counts = {c: len(list((OUT / sp / c).glob('*.jpg'))) for c in CLASSES}
    print(f'   • {sp.upper():<5} (Total {sum(counts.values()):4d}) : {counts}')


---
## ⚡ Tahap 3: Ekstraksi Fitur Spasial $8 \times 8 \times C$ (`InceptionV3` & `DenseNet169`)
Untuk mempercepat eksperimen **B1 (Baseline CNN)**, **B2 (Mutated CNN P6)**, dan **T1 (TLM-ViT Frozen P5)**, kita mengekstrak peta fitur (*feature map*) berukuran $8 \times 8 \times 2048$ (`InceptionV3`) dan $9 \times 9 \times 1664 \rightarrow$ di-*pool* atau langsung dipakai (`DenseNet169` pada input $299\times299$ menghasilkan $9\times9\times1664$ atau $8\times8\times1664$).

Dengan menyimpan peta fitur ini ke `.npy`, pelatihan model B1, B2, dan T1 hanya membutuhkan waktu **~15 detik per model**!


In [ ]:
BATCH_SIZE = 32

def make_ds(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        OUT / split,
        labels='inferred',
        label_mode='categorical',
        class_names=CLASSES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED
    ).prefetch(tf.data.AUTOTUNE)

BACKBONES = {
    'inc':   (tf.keras.applications.InceptionV3, tf.keras.applications.inception_v3.preprocess_input),
    'dense': (tf.keras.applications.DenseNet169, tf.keras.applications.densenet.preprocess_input),
}

# Cek apakah semua file fitur sudah diekstrak
expected_files = [FEAT / f'{tag}_{sp}.npy' for tag in BACKBONES for sp in ['train', 'val', 'test']]
if all(p.exists() for p in expected_files):
    print('✅ Seluruh fitur .npy sudah tersedia di memori!')
else:
    for tag, (Net, pre) in BACKBONES.items():
        print(f'⏳ Mengekstrak peta fitur spasial dari {tag.upper()}...')
        inp = layers.Input((*IMG_SIZE, 3))
        base_net = Net(include_top=False, weights='imagenet', input_tensor=pre(inp))
        # Pastikan dimensi spasial konsisten 8x8 agar PatchExtractor(patch=2) menghasilkan 16 patch (4x4)
        feat_out = base_net.output
        if feat_out.shape[1] != 8:
            feat_out = layers.Resizing(8, 8)(feat_out)
        extractor = models.Model(inp, feat_out)

        for sp in ['train', 'val', 'test']:
            ds = make_ds(sp, shuffle=False)
            X, Y = [], []
            for bx, by in ds:
                X.append(extractor(bx, training=False).numpy().astype('float16'))
                Y.append(by.numpy().astype('float32'))
            X = np.concatenate(X)
            Y = np.concatenate(Y)
            np.save(FEAT / f'{tag}_{sp}.npy', X)
            if not (FEAT / f'y_{sp}.npy').exists():
                np.save(FEAT / f'y_{sp}.npy', Y)
            print(f'   ✓ {tag}_{sp}.npy selesai -> shape: {X.shape}')
        del extractor, base_net
        tf.keras.backend.clear_session()

y_tr = np.load(FEAT / 'y_train.npy')
y_va = np.load(FEAT / 'y_val.npy')
y_te = np.load(FEAT / 'y_test.npy')
print(f'\n✅ Label siap: Train={y_tr.shape}, Val={y_va.shape}, Test={y_te.shape}')


---
## 🧩 Tahap 4: Definisi Layer Vision Transformer (`PatchExtractor` & `PatchEncoder`) & Evaluator
Di notebook tim sebelumnya, `class PatchExtractor` tidak terdefinisi sehingga memunculkan `NameError`. Di bawah ini adalah implementasi lengkap dan resmi sesuai **Paper P5 (Tabbakh & Barpanda, 2023)**:
- **`PatchExtractor`**: Memotong *feature map* CNN ($8 \times 8 \times C$) menjadi sekumpulan *patch* berukuran $p \times p$. Dengan $p=2$, dihasilkan $(8/2)^2 = 16$ *patch*.
- **`PatchEncoder`**: Memproyeksikan setiap *patch* ke dimensi vektor $D=64$ menggunakan *Linear Projection* (`Dense`) ditambah **Learnable Positional Embedding** agar informasi lokasi bagian daun tetap terjaga.
- **Transformer Encoder Block**: Terdiri dari `LayerNormalization` $\rightarrow$ `MultiHeadAttention` $\rightarrow$ Residual Add $\rightarrow$ `LayerNormalization` $\rightarrow$ MLP (`GELU`) $\rightarrow$ Residual Add.


In [ ]:
# ==============================================================================
# 1. CUSTOM LAYERS VISION TRANSFORMER (SESUAI PAPER P5 - TLMViT)
# ==============================================================================
class PatchExtractor(layers.Layer):
    """Memotong feature map (H x W x C) menjadi N patches berukuran (patch_size x patch_size)."""
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patch_dims = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dims])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'patch_size': self.patch_size})
        return cfg


class PatchEncoder(layers.Layer):
    """Proyeksi linear patch ke dimensi D + Positional Embedding."""
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patch) + self.pos_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


def build_vit_blocks(x, patch_size=2, dim=64, heads=4, blocks=4, mlp_units=(128, 64), drop=0.3):
    """Membangun rangkaian Transformer Encoder di atas feature map CNN (8x8xC)."""
    num_patches = (x.shape[1] // patch_size) ** 2
    x = PatchExtractor(patch_size)(x)
    x = PatchEncoder(num_patches, dim)(x)
    for _ in range(blocks):
        # Sub-layer 1: Multi-Head Self-Attention
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn = layers.MultiHeadAttention(num_heads=heads, key_dim=dim // heads, dropout=drop)(x1, x1)
        x2 = layers.Add()([attn, x])
        # Sub-layer 2: Feed-Forward MLP dengan aktivasi GELU
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        for u in mlp_units:
            x3 = layers.Dense(u, activation=tf.nn.gelu)(x3)
            x3 = layers.Dropout(drop)(x3)
        x = layers.Add()([x3, x2])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return layers.Flatten()(x)

print('✅ Layer PatchExtractor, PatchEncoder, dan build_vit_blocks berhasil didefinisikan!')


In [ ]:
# ==============================================================================
# 2. DICTIONARY PENYIMPANAN HASIL & FUNGSI EVALUASI STANDAR PAPER P6
# ==============================================================================
RES_PATH = DRIVE_BASE / 'RES_optimized.pkl'
RES = {}
if RES_PATH.exists():
    try:
        RES = pickle.load(open(RES_PATH, 'rb'))
        print(f'✅ Memuat checkpoint hasil sebelumnya dari Drive: {list(RES.keys())}')
    except Exception:
        RES = {}

def save_res():
    pickle.dump(RES, open(RES_PATH, 'wb'))
    pickle.dump(RES, open(FEAT / 'RES_new.pkl', 'wb'))

def evaluate_predictions(name, p_va, p_te, hist=None):
    """Menghitung metrik lengkap sesuai Tabel 13, 14, 15 Paper P6."""
    yt_va, yp_va = y_va.argmax(1), p_va.argmax(1)
    yt_te, yp_te = y_te.argmax(1), p_te.argmax(1)

    va_acc = accuracy_score(yt_va, yp_va)
    te_acc = accuracy_score(yt_te, yp_te)
    pr, rc, f1, _ = precision_recall_fscore_support(yt_va, yp_va, average='macro', zero_division=0)
    te_pr, te_rc, te_f1, _ = precision_recall_fscore_support(yt_te, yp_te, average='macro', zero_division=0)
    pr_c, rc_c, f1_c, _ = precision_recall_fscore_support(yt_va, yp_va, average=None, zero_division=0)

    cm = confusion_matrix(yt_te, yp_te)
    per_cls_correct = {c: (int(cm[i, i]), int(cm[i].sum())) for i, c in enumerate(CLASSES)}
    tot_correct = int(np.trace(cm))

    RES[name] = {
        'val_acc': va_acc, 'pr': pr, 'rc': rc, 'f1': f1,
        'test_acc': te_acc, 'test_pr': te_pr, 'test_rc': te_rc, 'test_f1': te_f1,
        'per_cls_val': {c: (pr_c[i], rc_c[i], f1_c[i]) for i, c in enumerate(CLASSES)},
        'per_cls': per_cls_correct, 'tot': tot_correct, 'cm': cm,
        'p_va': p_va, 'p_te': p_te, 'hist': hist
    }
    save_res()
    print(f'📊 [{name:<16}] Val Acc: {va_acc*100:6.2f}% | Val F1: {f1*100:6.2f}% || Test Benar: {tot_correct:3d}/305 ({te_acc*100:6.2f}%) | Test F1: {te_f1*100:6.2f}%')
    print(f'   └─ Per Kelas Test : ' + ', '.join(f'{c}: {v[0]}/{v[1]}' for c, v in per_cls_correct.items()))
    return RES[name]


---
## 🚀 Tahap 5: Pelatihan Bertahap (Ablation Study: `B1` $\rightarrow$ `B2` $\rightarrow$ `T1` $\rightarrow$ `T2`)
Kita melatih dan membandingkan 4 generasi arsitektur secara sistematis:
1. **Skema `B1` (Baseline CNN — Tabel 13 Paper P6):** Fitur CNN $\rightarrow$ `GAP` $\rightarrow$ `Softmax(4)` dengan Adagrad.
2. **Skema `B2` (Mutated CNN — Tabel 14 Paper P6):** Fitur CNN $\rightarrow$ `GAP` $\rightarrow$ `Dense` $\rightarrow$ `Dropout(0.5)` $\rightarrow$ `Softmax(4)`.
3. **Skema `T1` (TLM-ViT Frozen — Integrasi Paper P5):** Fitur CNN ($8\times8\times C$) $\rightarrow$ `Vision Transformer` $\rightarrow$ `MLP Head` $\rightarrow$ `Softmax(4)`.
4. **Skema `T2` (TLM-ViT 2-Stage End-to-End Fine-Tuning — Model Unggulan Kita):** Citra mentah + *Online Augmentation* $\rightarrow$ Stage 1 (Warmup ViT) $\rightarrow$ Stage 2 (Unfreeze Top Block CNN + ViT dengan `ReduceLROnPlateau`).


In [ ]:
# ==============================================================================
# EKSPERIMEN B1 (BASELINE CNN) & B2 (MUTATED CNN PAPER P6) & T1 (TLM-ViT FROZEN)
# ==============================================================================
def build_b1_or_b2(input_shape, hidden_layers=(), drop=0.5):
    inp = layers.Input(shape=input_shape)
    x = layers.GlobalAveragePooling2D()(inp)
    for h in hidden_layers:
        x = layers.Dense(h, activation='relu')(x)
        x = layers.Dropout(drop)(x)
    out = layers.Dense(len(CLASSES), activation='softmax')(x)
    return models.Model(inp, out)

def build_t1_tlmvit(input_shape, patch=2, dim=64, heads=4, blocks=4, mlp=(128, 64), top=(256,), drop=0.3):
    inp = layers.Input(shape=input_shape)
    x = build_vit_blocks(inp, patch_size=patch, dim=dim, heads=heads, blocks=blocks, mlp_units=mlp, drop=drop)
    for h in top:
        x = layers.Dense(h, activation=tf.nn.gelu)(x)
        x = layers.Dropout(drop)(x)
    out = layers.Dense(len(CLASSES), activation='softmax')(x)
    return models.Model(inp, out)

def train_on_cached_features(tag, name, model_fn, lr, opt_name='adagrad', batch=16, epochs=35, patience=7):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    Xtr = np.load(FEAT / f'{tag}_train.npy').astype('float32')
    Xva = np.load(FEAT / f'{tag}_val.npy').astype('float32')
    Xte = np.load(FEAT / f'{tag}_test.npy').astype('float32')

    model = model_fn(Xtr.shape[1:])
    opt = tf.keras.optimizers.Adagrad(learning_rate=lr) if opt_name == 'adagrad' else tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    cb = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=patience, restore_best_weights=True, mode='max')]
    hist = model.fit(Xtr, y_tr, validation_data=(Xva, y_va), epochs=epochs, batch_size=batch, callbacks=cb, verbose=0)

    p_va = model.predict(Xva, verbose=0)
    p_te = model.predict(Xte, verbose=0)
    return evaluate_predictions(name, p_va, p_te, hist.history)

print('--- [1/3] Melatih Skema B1 (Baseline CNN - Tabel 13 Paper P6) ---')
train_on_cached_features('inc',   'B1 Inception',   lambda s: build_b1_or_b2(s, ()), lr=0.01857, opt_name='adagrad')
train_on_cached_features('dense', 'B1 DenseNet169', lambda s: build_b1_or_b2(s, ()), lr=0.01857, opt_name='adagrad')

print('\n--- [2/3] Melatih Skema B2 (Mutated CNN - Tabel 14 Paper P6) ---')
train_on_cached_features('inc',   'B2 Inception',   lambda s: build_b1_or_b2(s, (1024, 512), 0.5), lr=0.01857, opt_name='adagrad')
train_on_cached_features('dense', 'B2 DenseNet169', lambda s: build_b1_or_b2(s, (512,),      0.5), lr=0.01857, opt_name='adagrad')

print('\n--- [3/3] Melatih Skema T1 (TLM-ViT Frozen - Metode Paper P5) ---')
train_on_cached_features('inc',   'T1 Inception',   lambda s: build_t1_tlmvit(s, patch=2, dim=64, heads=4, blocks=4), lr=3e-4, opt_name='adam', batch=32, epochs=50, patience=10)
train_on_cached_features('dense', 'T1 DenseNet169', lambda s: build_t1_tlmvit(s, patch=2, dim=64, heads=4, blocks=4), lr=3e-4, opt_name='adam', batch=32, epochs=50, patience=10)


In [ ]:
# ==============================================================================
# EKSPERIMEN T2 (TLM-ViT 2-STAGE END-TO-END FINE-TUNING — MODEL UNGGULAN KITA)
# ==============================================================================
DS_TR = make_ds('train', shuffle=True)
DS_VA = make_ds('val',   shuffle=False)
DS_TE = make_ds('test',  shuffle=False)

# Unfreeze mulai dari blok konvolusi atas agar fitur beradaptasi ke daun pisang
UNFREEZE_FROM = {'inc': 'mixed7', 'dense': 'conv5_block1_0_bn'}

def run_t2_finetune(tag, name, ep_stage1=12, ep_stage2=25, lr_stage1=3e-4, lr_stage2=1e-4, force_retrain=False):
    if name in RES and not force_retrain:
        print(f'ℹ️ {name} sudah ada di checkpoint, menampilkan hasil tersimpan:')
        r = RES[name]
        print(f'📊 [{name:<16}] Val Acc: {r["val_acc"]*100:6.2f}% | Val F1: {r["f1"]*100:6.2f}% || Test Benar: {r["tot"]:3d}/305 ({r["test_acc"]*100:6.2f}%)')
        return r

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    Net, pre = BACKBONES[tag]

    inp = layers.Input((*IMG_SIZE, 3))
    # Augmentasi online ringan saat training end-to-end
    x = layers.RandomFlip('horizontal', seed=SEED)(inp)
    x = layers.RandomRotation(0.05, seed=SEED)(x)
    x = layers.RandomZoom(0.08, seed=SEED)(x)

    base = Net(include_top=False, weights='imagenet', input_tensor=pre(x))
    base.trainable = False

    feat = base.output
    if feat.shape[1] != 8:
        feat = layers.Resizing(8, 8)(feat)

    z = build_vit_blocks(feat, patch_size=2, dim=64, heads=4, blocks=4, mlp_units=(128, 64), drop=0.3)
    z = layers.Dense(256, activation=tf.nn.gelu)(z)
    z = layers.Dropout(0.3)(z)
    out = layers.Dense(len(CLASSES), activation='softmax')(z)
    model = models.Model(inp, out)

    print(f'\n🔥 [{name}] Stage 1: Warmup ViT Head ({ep_stage1} epochs, Backbone Frozen)...')
    model.compile(optimizer=tf.keras.optimizers.Adam(lr_stage1), loss='categorical_crossentropy', metrics=['accuracy'])
    cb1 = [tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max')]
    h1 = model.fit(DS_TR, validation_data=DS_VA, epochs=ep_stage1, callbacks=cb1, verbose=1)

    # Stage 2: Unfreeze blok konvolusi atas CNN (BatchNorm tetap dibekukan agar statistik stabil)
    print(f'🔥 [{name}] Stage 2: Joint Fine-Tuning Top Block CNN + ViT ({ep_stage2} epochs)...')
    unfreeze_flag = False
    for lyr in base.layers:
        if lyr.name == UNFREEZE_FROM[tag]:
            unfreeze_flag = True
        if unfreeze_flag and not isinstance(lyr, layers.BatchNormalization):
            lyr.trainable = True

    model.compile(optimizer=tf.keras.optimizers.Adam(lr_stage2), loss='categorical_crossentropy', metrics=['accuracy'])
    cb2 = [
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]
    h2 = model.fit(DS_TR, validation_data=DS_VA, epochs=ep_stage2, callbacks=cb2, verbose=1)

    hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
    p_va = model.predict(DS_VA, verbose=0)
    p_te = model.predict(DS_TE, verbose=0)

    # Simpan bobot model terbaik ke Drive
    w_file = name.replace(' ', '_') + '.weights.h5'
    model.save_weights(str(DRIVE_BASE / w_file))
    return evaluate_predictions(name, p_va, p_te, hist)

run_t2_finetune('inc',   'T2 Inception')
run_t2_finetune('dense', 'T2 DenseNet169')


---
## 🏆 Tahap 6: Deep Ensemble Strategy (`Unweighted` vs `Optimal Weighted`) & Perbandingan Paper P6
Di bagian ini kita menerapkan:
1. **Standard Average Ensemble (Unweighted $0.5 : 0.5$)** sesuai **Algorithm 1 Paper P6**:
   $$P_{\text{ens}}(y \mid x) = \frac{P_{\text{InceptionV3}}(y \mid x) + P_{\text{DenseNet169}}(y \mid x)}{2}$$
2. **Optimal Weighted Soft-Voting Ensemble** sesuai **Persamaan (13) Paper P6**:
   $$P_{\text{weighted}}(y \mid x) = w_1 \cdot P_{\text{InceptionV3}}(y \mid x) + w_2 \cdot P_{\text{DenseNet169}}(y \mid x)$$
   di mana bobot $(w_1, w_2)$ dicari secara otomatis berdasarkan **akurasi & F1 tertinggi pada Validation Set** (tanpa melihat Test Set), sehingga ketika satu model jauh lebih kuat (`DenseNet169`), prediksinya tidak terdistorsi oleh model yang lebih lemah.
3. **Super-Ensemble (`B2 + T2`):** Menggabungkan kekuatan representasi *Global Average Pooling* (`B2`) dan *Spatial Self-Attention ViT* (`T2`).


In [ ]:
# ==============================================================================
# 1. UNWEIGHTED ENSEMBLE (0.5 : 0.5) & OPTIMAL WEIGHTED ENSEMBLE
# ==============================================================================
def run_ensemble(ens_name, model_a, model_b, weighted=False):
    pa_va, pb_va = RES[model_a]['p_va'], RES[model_b]['p_va']
    pa_te, pb_te = RES[model_a]['p_te'], RES[model_b]['p_te']

    if not weighted:
        w_best = 0.5
    else:
        # Cari bobot w_a di Validation Set (grid 0.05 s/d 0.95)
        yt_va = y_va.argmax(1)
        best_score, w_best = -1.0, 0.5
        for w in np.linspace(0.05, 0.95, 19):
            p_mix = w * pa_va + (1.0 - w) * pb_va
            acc = accuracy_score(yt_va, p_mix.argmax(1))
            if acc > best_score:
                best_score, w_best = acc, w
        print(f'   💡 Bobot Optimal Validasi untuk [{ens_name}]: {w_best:.2f}*{model_a} + {1-w_best:.2f}*{model_b}')

    p_va = w_best * pa_va + (1.0 - w_best) * pb_va
    p_te = w_best * pa_te + (1.0 - w_best) * pb_te
    return evaluate_predictions(ens_name, p_va, p_te, None)

print('=== A. ENSEMBLE RATA-RATA STANDAR (0.5 : 0.5) ===')
run_ensemble('Ens B1', 'B1 Inception', 'B1 DenseNet169', weighted=False)
run_ensemble('Ens B2', 'B2 Inception', 'B2 DenseNet169', weighted=False)
run_ensemble('Ens T1', 'T1 Inception', 'T1 DenseNet169', weighted=False)
run_ensemble('Ens T2', 'T2 Inception', 'T2 DenseNet169', weighted=False)

print('\n=== B. OPTIMAL WEIGHTED ENSEMBLE & HYBRID SUPER-ENSEMBLE ===')
run_ensemble('Ens T2 (Weighted)', 'T2 Inception', 'T2 DenseNet169', weighted=True)

# Super-Ensemble: Menggabungkan 4 Model (B2 Inc + B2 Dense + T2 Inc + T2 Dense)
p_va_super = 0.15 * RES['B2 Inception']['p_va'] + 0.20 * RES['B2 DenseNet169']['p_va'] + 0.20 * RES['T2 Inception']['p_va'] + 0.45 * RES['T2 DenseNet169']['p_va']
p_te_super = 0.15 * RES['B2 Inception']['p_te'] + 0.20 * RES['B2 DenseNet169']['p_te'] + 0.20 * RES['T2 Inception']['p_te'] + 0.45 * RES['T2 DenseNet169']['p_te']
evaluate_predictions('Super Ens (B2+T2)', p_va_super, p_te_super, None)


In [ ]:
# ==============================================================================
# 2. TABEL KOMPARASI LENGKAP: HASIL KITA VS PAPER RUJUKAN (P6 - IEEE ACCESS 2024)
# ==============================================================================
PAPER_BENCHMARK = {
    'Paper P6: Orig InceptionV3 (Tbl 13)':     (0.7562, 0.7800, 0.7600, 0.7600, None),
    'Paper P6: Orig DenseNet169 (Tbl 13)':     (0.8875, 0.8900, 0.8900, 0.8900, None),
    'Paper P6: Mutated InceptionV3 (Tbl 14)':  (0.9562, 0.8800, 0.8700, 0.8600, 265),
    'Paper P6: Mutated DenseNet169 (Tbl 14)':  (0.9775, 0.9200, 0.9200, 0.9200, 281),
    'Paper P6: Proposed IncepV3Dense (Tbl 14)':(0.9862, 0.9400, 0.9400, 0.9300, 285),
}

rows = []
# Masukkan benchmark Paper P6
for k, (va, pr, rc, f1, tot) in PAPER_BENCHMARK.items():
    rows.append({
        'Kelompok': '1. Benchmark Paper P6 (2024)',
        'Model': k,
        'Val Acc (%)': round(va * 100, 2),
        'Val Prec (%)': round(pr * 100, 2),
        'Val Rec (%)': round(rc * 100, 2),
        'Val F1 (%)': round(f1 * 100, 2),
        'Test Benar (/305)': f'{tot}/305' if tot else '-',
        'Test Acc (%)': round(tot / 305 * 100, 2) if tot else None,
    })

# Masukkan hasil eksperimen kita
ORDER = [
    'B1 Inception', 'B1 DenseNet169', 'Ens B1',
    'B2 Inception', 'B2 DenseNet169', 'Ens B2',
    'T1 Inception', 'T1 DenseNet169', 'Ens T1',
    'T2 Inception', 'T2 DenseNet169', 'Ens T2', 'Ens T2 (Weighted)', 'Super Ens (B2+T2)'
]
for k in ORDER:
    if k in RES:
        r = RES[k]
        rows.append({
            'Kelompok': '2. Eksperimen Kita (P5 + P6)',
            'Model': k,
            'Val Acc (%)': round(r['val_acc'] * 100, 2),
            'Val Prec (%)': round(r['pr'] * 100, 2),
            'Val Rec (%)': round(r['rc'] * 100, 2),
            'Val F1 (%)': round(r['f1'] * 100, 2),
            'Test Benar (/305)': f"{r['tot']}/305",
            'Test Acc (%)': round(r['test_acc'] * 100, 2),
        })

df_comp = pd.DataFrame(rows)
display(df_comp)

# Simpan ke CSV di Google Drive
df_comp.to_csv(DRIVE_BASE / 'Tabel_Komparasi_Paper_vs_Eksperimen.csv', index=False)
print(f'\n💾 Tabel berhasil disimpan ke: {DRIVE_BASE / "Tabel_Komparasi_Paper_vs_Eksperimen.csv"}')


In [ ]:
# ==============================================================================
# 3. VISUALISASI CONFUSION MATRIX & GRAFIK PENINGKATAN AKURASI TEST (305 CITRA)
# ==============================================================================
plot_models = ['Ens B2', 'Ens T1', 'T2 DenseNet169', 'Ens T2', 'Ens T2 (Weighted)', 'Super Ens (B2+T2)']
plot_models = [m for m in plot_models if m in RES]

fig, axes = plt.subplots(1, len(plot_models), figsize=(4.2 * len(plot_models), 3.8))
if len(plot_models) == 1:
    axes = [axes]
for ax, m in zip(axes, plot_models):
    cm = RES[m]['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=CLASSES, yticklabels=CLASSES, cbar=False, ax=ax)
    ax.set_title(f"{m}\nBenar: {RES[m]['tot']}/305 ({RES[m]['test_acc']*100:.2f}%)", fontweight='bold', fontsize=10)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')
plt.suptitle('Confusion Matrix pada 305 Citra Uji (Paper P6 IncepV3Dense = 285/305 atau 93.44%)', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(DRIVE_BASE / 'Confusion_Matrix_Comparison.png', dpi=180, bbox_inches='tight')
plt.show()

# Bar Chart Perbandingan Langsung Akurasi Test vs Paper P6
plt.figure(figsize=(11, 5))
bar_labels = ['Paper P6:\nMut. IncV3', 'Paper P6:\nMut. Dense169', 'Paper P6:\nIncepV3Dense\n(285/305)'] + [f"{m}\n({RES[m]['tot']}/305)" for m in plot_models]
bar_vals   = [265/305*100, 281/305*100, 285/305*100] + [RES[m]['test_acc']*100 for m in plot_models]
bar_colors = ['#94a3b8', '#64748b', '#f59e0b'] + ['#10b981']*len(plot_models)

bars = plt.bar(bar_labels, bar_vals, color=bar_colors, edgecolor='black', width=0.6)
plt.axhline(285/305*100, color='#d97706', linestyle='--', linewidth=1.8, label='Batas Terbaik Paper P6 (93.44% / 285 benar)')
plt.ylim(80, 101.5)
plt.ylabel('Test Accuracy pada 305 Citra Uji (%)', fontweight='bold')
plt.title('Perbandingan Akurasi Data Uji (305 Citra): Paper P6 (2024) vs Deep Ensemble TLM-ViT Kita', fontweight='bold', fontsize=12)
for b, v in zip(bars, bar_vals):
    plt.text(b.get_x() + b.get_width()/2, v + 0.4, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=9)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_BASE / 'Bar_Chart_Paper_vs_Ours.png', dpi=180, bbox_inches='tight')
plt.show()
